# Analyze the segmentation and alignment quality of the Xenium patches (patch_metrics.csv) for a given dataset_id

In [ ]:
import sys
from pathlib import Path
import os
import json
import io
import zipfile
import pandas as pd
import spatialdata as sd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import to_rgba
from scipy.sparse import csr_matrix
import h5py

# Add the project root to sys.path so that 'src' can be imported
project_root = Path.cwd().parents[2]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.constants import get_full_path

## Helper functions

In [ ]:
def plot_all_metrics(df_metrics):

    metrics = ['Dice', 'Jaccard', 'bPQ']

    plt.figure(figsize=(9, 3))

    for i, metric in enumerate(metrics, 1):
        plt.subplot(1, 3, i)
        sns.histplot(df_metrics.loc[df_metrics["Dice"] != -1, metric], bins=30, kde=True, color='#6e7f94')
        plt.title(f'{metric}')
        plt.xlabel(metric)
        plt.ylabel('Frequency')

    plt.tight_layout()
    plt.show()

In [ ]:
def load_imgfile(img_zip_path, img_name):
    with zipfile.ZipFile(img_zip_path, 'r') as img_zip:
        img_data = img_zip.read(img_name)
        img = Image.open(io.BytesIO(img_data))
    return np.array(img).astype(np.uint8)


def load_maskfile(mask_zip_path, mask_name):
        
    with zipfile.ZipFile(mask_zip_path, 'r') as mask_zip:
        
        with mask_zip.open(mask_name) as mask_file:
            # Load the .npz file using numpy
            data = np.load(io.BytesIO(mask_file.read()), allow_pickle=True)
            
            # Reconstruct sparse matrices
            inst_map_sparse = csr_matrix(
                (data["inst_map_data"], data["inst_map_indices"], data["inst_map_indptr"]),
                shape=data["inst_map_shape"]
            )
            type_map_sparse = csr_matrix(
                (data["type_map_data"], data["type_map_indices"], data["type_map_indptr"]),
                shape=data["type_map_shape"]
            )

            # Convert sparse matrices to dense arrays
            inst_map = inst_map_sparse.toarray().astype(np.int32)
            type_map = type_map_sparse.toarray().astype(np.int32)
    return inst_map, type_map

In [ ]:
class H5FileSingleton:
    _instances = {}

    @classmethod
    def get_instance(cls, file_path):
        if file_path not in cls._instances:
            cls._instances[file_path] = h5py.File(file_path, 'r')
        return cls._instances[file_path]

    @classmethod
    def close_all(cls):
        for file_path, h5_file in cls._instances.items():
            if h5_file is not None:
                h5_file.close()
        cls._instances.clear()

In [ ]:
def load_sparse_h5(file_path, image_name):
    """
    Load a sparse CSR matrix for a specific key (image_name) from a shared HDF5 file.
    
    Parameters:
        file_path (str): Path to the global HDF5 file.
        image_name (str): Key for the specific patch in the file.
    
    Returns:
        csr_matrix: Reconstructed sparse matrix.
    """
    h5_file = H5FileSingleton.get_instance(file_path)
    data = h5_file[f"{image_name}_data"][:]
    indices = h5_file[f"{image_name}_indices"][:]
    indptr = h5_file[f"{image_name}_indptr"][:]
    shape = tuple(h5_file[f"{image_name}_shape"][:])  # Ensure shape is a tuple
    return csr_matrix((data, indices, indptr), shape=shape)

In [ ]:
def plot_check_seg(df_metrics, metric, low_threshold, high_threshold, n_plot, img_zip_path, mask_zip_path, inst_map_pred_path, save_folder=None):

    patch_ids_selected = df_metrics.loc[(df_metrics[metric] > low_threshold) & (df_metrics[metric] < high_threshold), "patch_id"]

    patch_ids = patch_ids_selected.sample(min(n_plot, len(patch_ids_selected)))

    for id in patch_ids:

        row = df_metrics[df_metrics['patch_id'] == id].iloc[0]
        dice_score = row['Dice']
        jaccard_score = row['Jaccard']
        pq_score = row['bPQ']

        # Mask from ST
        inst_map, type_map = load_maskfile(mask_zip_path, f"{id + '.npz'}")

        unique_objects = np.unique(inst_map)[np.unique(inst_map) != 0]  # Exclude the background (0)
        num_objects = len(unique_objects)
        cmap = plt.get_cmap('hsv', num_objects)
        idx2color_mask = {obj: to_rgba(cmap(i), alpha=1.0) for i, obj in enumerate(unique_objects, start=1)}
        idx2color_mask[0] = (0, 0, 0, 0)  # Transparent background for value 0

        # H&E patch
        image = load_imgfile(img_zip_path, f"{id + '.png'}")

        # The predicted segmentation
        inst_map_pred = load_sparse_h5(inst_map_pred_path, f"{id}")
        inst_map_pred = inst_map_pred.toarray().astype(np.int32)

        unique_objects = np.unique(inst_map_pred)[np.unique(inst_map_pred) != 0]  # Exclude the background (0)
        num_objects = len(unique_objects)
        cmap = plt.get_cmap('hsv', num_objects)
        idx2color_pred = {obj: to_rgba(cmap(i), alpha=1.0) for i, obj in enumerate(unique_objects, start=1)}
        idx2color_pred[0] = (0, 0, 0, 0)  # Transparent background for value 0

        plt.figure(figsize=(5, 2))
        # Plot H&E patch
        plt.subplot(1,3,1)
        plt.imshow(image)
        plt.tick_params(axis='both', which='major', labelsize=6)
        plt.title(f'{id}')
        # Plot H&E patch with mask
        plt.subplot(1,3,2)
        plt.imshow(image)
        mask_plotting = np.array([[idx2color_mask[idx] for idx in row] for row in inst_map])
        plt.imshow(mask_plotting, alpha=0.4)
        plt.tick_params(axis='both', which='major', labelsize=6)
        plt.title('From ST')
        # Plot H&E patch with predicted mask
        plt.subplot(1,3,3)
        plt.imshow(image)
        pred_plotting = np.array([[idx2color_pred[idx] for idx in row] for row in inst_map_pred])
        plt.imshow(pred_plotting, alpha=0.4)
        plt.tick_params(axis='both', which='major', labelsize=6)
        plt.title('CellViT prediction')

        plt.figtext(0.5, 0.02, f'Dice: {dice_score:.3f}, Jaccard: {jaccard_score:.3f}, bPQ: {pq_score:.3f}', 
                    wrap=True, horizontalalignment='center', fontsize=10)
        
        plt.tight_layout()

        if save_folder:
            plt.savefig(os.path.join(save_folder, f'{id}_{metric}-{row[metric]}.png'), dpi=300)

        plt.show()

In [ ]:
def plot_patch_seg(patch_id, df_metrics, img_zip_path, mask_zip_path, inst_map_pred_path, save_folder=None):

    row = df_metrics[df_metrics['patch_id'] == patch_id].iloc[0]
    dice_score = row['Dice']
    jaccard_score = row['Jaccard']
    pq_score = row['bPQ']

    # Mask from ST
    inst_map, type_map = load_maskfile(mask_zip_path, f"{patch_id}.npz")

    unique_objects = np.unique(inst_map)[np.unique(inst_map) != 0]  # Exclude the background (0)
    num_objects = len(unique_objects)
    cmap = plt.get_cmap('hsv', num_objects)
    idx2color_mask = {obj: to_rgba(cmap(i), alpha=1.0) for i, obj in enumerate(unique_objects, start=1)}
    idx2color_mask[0] = (0, 0, 0, 0)  # Transparent background for value 0

    # H&E patch
    image = load_imgfile(img_zip_path, f"{patch_id}.png")

    # The predicted segmentation
    inst_map_pred = load_sparse_h5(inst_map_pred_path, f"{patch_id}")
    inst_map_pred = inst_map_pred.toarray().astype(np.int32)

    unique_objects = np.unique(inst_map_pred)[np.unique(inst_map_pred) != 0]  # Exclude the background (0)
    num_objects = len(unique_objects)
    cmap = plt.get_cmap('hsv', num_objects)
    idx2color_pred = {obj: to_rgba(cmap(i), alpha=1.0) for i, obj in enumerate(unique_objects, start=1)}
    idx2color_pred[0] = (0, 0, 0, 0)  # Transparent background for value 0

    plt.figure(figsize=(5, 2))
    # Plot H&E patch
    plt.subplot(1,3,1)
    plt.imshow(image)
    plt.tick_params(axis='both', which='major', labelsize=6)
    plt.title(f'{patch_id}')
    # Plot H&E patch with mask
    plt.subplot(1,3,2)
    plt.imshow(image)
    mask_plotting = np.array([[idx2color_mask[idx] for idx in row] for row in inst_map])
    plt.imshow(mask_plotting, alpha=0.4)
    plt.tick_params(axis='both', which='major', labelsize=6)
    plt.title('From ST')
    # Plot H&E patch with predicted mask
    plt.subplot(1,3,3)
    plt.imshow(image)
    pred_plotting = np.array([[idx2color_pred[idx] for idx in row] for row in inst_map_pred])
    plt.imshow(pred_plotting, alpha=0.4)
    plt.tick_params(axis='both', which='major', labelsize=6)
    plt.title('CellViT prediction')

    plt.figtext(0.5, 0.02, f'Dice: {dice_score:.3f}, Jaccard: {jaccard_score:.3f}, bPQ: {pq_score:.3f}', 
                wrap=True, horizontalalignment='center', fontsize=10)
    
    plt.tight_layout()

    if save_folder:
        plt.savefig(os.path.join(save_folder, f'{patch_id}_check_seg.png'), dpi=300)

    plt.show()

## ds_ct1_20x_256_o112

In [ ]:
dataset_id = 'ds_ct1_20x_256_o112'

In [ ]:
output_cellvit_folder = get_full_path('train', 'datasets', dataset_id, 'do_patch_eval', as_str=True)
images_path = get_full_path('train', 'datasets', dataset_id, 'images.zip', as_str=True)
labels_path = get_full_path('train', 'datasets', dataset_id, 'labels.zip', as_str=True)
patch_metrics_path = get_full_path('train', 'datasets', dataset_id, 'patch_metrics.csv', as_str=True)

### All slides

In [ ]:
df_metrics = pd.read_csv(patch_metrics_path)
df_metrics

In [ ]:
plot_all_metrics(df_metrics)

### Analyze for breast_s0

In [ ]:
slide_id = "breast_s0"

In [ ]:
plot_all_metrics(df_metrics[df_metrics['slide_id'] == slide_id])

In [ ]:
plot_check_seg(df_metrics=df_metrics[df_metrics['slide_id'] == slide_id], 
            metric='Jaccard', 
            low_threshold=0.2, 
            high_threshold=0.6, 
            n_plot=2,
            img_zip_path=images_path,
            mask_zip_path=labels_path,
            inst_map_pred_path=os.path.join(output_cellvit_folder, 'run1', 'inference_instance_map_predictions.h5'),
            save_folder=None)

In [ ]:
plot_patch_seg(patch_id='breast_s0_156', 
            df_metrics=df_metrics[df_metrics['slide_id'] == slide_id], 
            img_zip_path=images_path,
            mask_zip_path=labels_path,
            inst_map_pred_path=os.path.join(output_cellvit_folder, 'run1', 'inference_instance_map_predictions.h5'),
            save_folder=None)

### Choose the metrics and the threshold

In [ ]:
# Loop to look at Dice for 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9
for threshold in np.arange(0.1, 1.0, 0.1):
    print(f"Threshold: {threshold}")
    plot_check_seg(df_metrics=df_metrics, 
                metric='Dice', 
                low_threshold=threshold, 
                high_threshold=threshold+0.1, 
                n_plot=2,
                img_zip_path=images_path,
                mask_zip_path=labels_path,
                inst_map_pred_path=os.path.join(output_cellvit_folder, 'run1', 'inference_instance_map_predictions.h5'),
                save_folder=None)

In [ ]:
# Same for Jaccard
for threshold in np.arange(0.1, 1.0, 0.1):
    print(f"Threshold: {threshold}")
    plot_check_seg(df_metrics=df_metrics, 
                metric='Jaccard', 
                low_threshold=threshold, 
                high_threshold=threshold+0.1, 
                n_plot=2,
                img_zip_path=images_path,
                mask_zip_path=labels_path,
                inst_map_pred_path=os.path.join(output_cellvit_folder, 'run1', 'inference_instance_map_predictions.h5'),
                save_folder=None)

In [ ]:
# Same for bPQ but regarding low thresholds
for threshold in np.arange(0.1, 0.5, 0.05):
    print(f"Threshold: {threshold}")
    plot_check_seg(df_metrics=df_metrics, 
                metric='bPQ', 
                low_threshold=threshold, 
                high_threshold=threshold+0.05, 
                n_plot=2,
                img_zip_path=images_path,
                mask_zip_path=labels_path,
                inst_map_pred_path=os.path.join(output_cellvit_folder, 'run1', 'inference_instance_map_predictions.h5'),
                save_folder=None)

### Explain the pick at 0

In [ ]:
# Get list of patch_ids with bqp < 0.05 and Dice > 0.5
low_bpq_high_dice = df_metrics[(df_metrics['bPQ'] < 0.05) & (df_metrics['Dice'] > 0.5)]['patch_id'].tolist()

In [ ]:
# Plot low_bpq_high_dice patches (select only 20 randomly)
low_bpq_high_dice_sample = np.random.choice(low_bpq_high_dice, size=min(20, len(low_bpq_high_dice)), replace=False)
for patch_id in low_bpq_high_dice_sample:
    plot_patch_seg(patch_id=patch_id, 
                df_metrics=df_metrics, 
                img_zip_path=images_path,
                mask_zip_path=labels_path,
                inst_map_pred_path=os.path.join(output_cellvit_folder, 'run1', 'inference_instance_map_predictions.h5'),
                save_folder=None)